# Comparacion CatBoost con parametros del grid search

Este notebook entrena el mismo modelo con los mejores parametros encontrados en el grid search y compara dos escenarios:

1. Target normalizado con `log1p(price)`.
2. Target sin normalizar.


In [ ]:
import importlib.util
import subprocess
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

if importlib.util.find_spec('catboost') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'catboost'])

from catboost import CatBoostRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

pd.set_option('display.max_columns', None)
sns.set(style='whitegrid')

In [ ]:
DATA_PATH = '../data/raw/kc_house_data.csv'
TARGET = 'price'
RANDOM_STATE = 42
TEST_SIZE = 0.2

# Mejores parametros reportados por el grid search previo.
PARAMS_GRID_SEARCH = {
    'iterations': 100,
    'learning_rate': 0.05,
    'depth': 6,
    'l2_leaf_reg': 3,
    'loss_function': 'RMSE',
    'eval_metric': 'RMSE',
    'random_seed': RANDOM_STATE,
    'verbose': 0,
    'allow_writing_files': False,
}

COLUMNAS_ELIMINAR = [
    'waterfront', 'data_modelod_fabricacion', 'sqft_basement', 'bathrooms',
    'condition', 'floors', 'bedrooms', 'tiene_renov', 'data_modelod_renovacion'
]

COLUMNAS_BASE_A_ELIMINAR = [
    'id', 'date', 'yr_built', 'yr_renovated', 'sqft_living15', 'sqft_lot15'
]

COLUMNAS_CATEGORICAS = ['view', 'grade', 'zipcode']

In [ ]:
def obtener_metricas(y_true, y_pred):
    return {
        'MAE': mean_absolute_error(y_true, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_true, y_pred)),
        'R2': r2_score(y_true, y_pred),
    }


def preparar_datos(df):
    data_modelo = df.copy()
    anio_actual = pd.Timestamp.today().year

    data_modelo['data_modelod_fabricacion'] = anio_actual - data_modelo['yr_built']
    data_modelo['data_modelod_renovacion'] = np.where(
        data_modelo['yr_renovated'] > 0,
        anio_actual - data_modelo['yr_renovated'],
        0,
    )
    data_modelo['tiene_renov'] = np.where(
        data_modelo['data_modelod_renovacion'] == 0,
        'No',
        'Si',
    )

    data_modelo = data_modelo.drop(columns=COLUMNAS_BASE_A_ELIMINAR, errors='ignore')
    data_modelo = data_modelo.drop(columns=COLUMNAS_ELIMINAR, errors='ignore')

    cat_features = [col for col in COLUMNAS_CATEGORICAS if col in data_modelo.columns]
    for col in cat_features:
        data_modelo[col] = data_modelo[col].astype(str)

    X = data_modelo.drop(columns=[TARGET])
    y = data_modelo[TARGET].copy()

    return X, y, cat_features


def entrenar_catboost(nombre, X_train, X_test, y_train, y_test, cat_features, usar_target_logaritmico):
    if usar_target_logaritmico:
        y_train_modelo = np.log1p(y_train.to_numpy())
        y_test_modelo = np.log1p(y_test.to_numpy())
    else:
        y_train_modelo = y_train
        y_test_modelo = y_test

    modelo = CatBoostRegressor(**PARAMS_GRID_SEARCH)
    modelo.fit(
        X_train,
        y_train_modelo,
        cat_features=cat_features,
        eval_set=(X_test, y_test_modelo),
        use_best_model=True,
    )

    y_pred_modelo = modelo.predict(X_test)
    y_pred = np.expm1(y_pred_modelo) if usar_target_logaritmico else y_pred_modelo

    return {
        'modelo': nombre,
        'target_logaritmico': usar_target_logaritmico,
        **obtener_metricas(y_test, y_pred),
    }, modelo, pd.DataFrame({'y_real': y_test, 'y_pred': y_pred}, index=y_test.index)

In [ ]:
df = pd.read_csv(DATA_PATH)
X, y, cat_features = preparar_datos(df)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
)

metricas_log, modelo_log, predicciones_log = entrenar_catboost(
    nombre='grid_search_target_log1p',
    X_train=X_train,
    X_test=X_test,
    y_train=y_train,
    y_test=y_test,
    cat_features=cat_features,
    usar_target_logaritmico=True,
)

metricas_sin_log, modelo_sin_log, predicciones_sin_log = entrenar_catboost(
    nombre='grid_search_target_original',
    X_train=X_train,
    X_test=X_test,
    y_train=y_train,
    y_test=y_test,
    cat_features=cat_features,
    usar_target_logaritmico=False,
)

comparacion_metricas = pd.DataFrame([metricas_log, metricas_sin_log]).set_index('modelo')
comparacion_metricas

In [ ]:
mejor_modelo = comparacion_metricas.sort_values(
    by=['R2', 'MAE', 'RMSE'],
    ascending=[False, True, True],
).head(1)

mejor_modelo

In [ ]:
metricas_plot = comparacion_metricas.reset_index().melt(
    id_vars='modelo',
    value_vars=['MAE', 'RMSE', 'R2'],
    var_name='metrica',
    value_name='valor',
)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, metrica in zip(axes, ['MAE', 'RMSE', 'R2']):
    data_plot = metricas_plot[metricas_plot['metrica'] == metrica]
    sns.barplot(data=data_plot, x='modelo', y='valor', ax=ax, palette='Blues_r')
    ax.set_title(metrica)
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.show()